In [1]:
import pandas as pd

In [3]:
df = pd.read_csv(
    "../data/interim/Net_electricity_generation.csv",
    skiprows=4
)

df.head(10)

,description,units,source key,2001,2002,2003,2004,2005,2006,2007,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Net generation for all fuels (utility-scale),thousand megawatthours,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,United States,thousand megawatthours,ELEC.GEN.ALL-US-99.A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,United States : all sectors,thousand megawatthours,ELEC.GEN.ALL-US-99.A,3736644,3858452,3883185,3970555,4055423,4064702,4156745,...,4077574,4035443,4180988,4130574,4009767,4109699,4230668,4183270,4308634,4429502
3,United States : electric power,NaN,ELEC.GEN.ALL-US-98.A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,United States : electric utility,thousand megawatthours,ELEC.GEN.ALL-US-1.A,2629946,2549457,2462281,2505231,2474846,2483656,2504131,...,2305887,2275539,2339960,2268723,2170316,2211643,2229605,2189614,2256336,2303765
5,United States : independent power producers,thousand megawatthours,ELEC.GEN.ALL-US-94.A,950107,1149001,1258879,1303129,1427346,1424421,1501212,...,1613090,1603086,1680917,1699625,1683340,1745538,1844282,1838927,1900291,1971436
6,United States : all commercial,thousand megawatthours,ELEC.GEN.ALL-US-96.A,7416,7415,7496,8270,8492,8371,8273,...,12706,13060,13312,13689,13046,12768,16737,16066,15258,15917
7,United States : all industrial,thousand megawatthours,ELEC.GEN.ALL-US-97.A,149175,152580,154530,153925,144739,148254,143128,...,145890,143758,146798,148537,143064,139750,140043,138664,136749,138383
8,New England,thousand megawatthours,ELEC.GEN.ALL-NEW-99.A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,New England : all sectors,thousand megawatthours,ELEC.GEN.ALL-NEW-99.A,116591,124340,130148,133518,136149,132211,132526,...,107725,105234,105548,99997,96809,103089,105612,102611,109310,112611


In [4]:
#keep only commercial and industrial rows
df_clean = df[
    df["description"].str.contains(r": all commercial|: all industrial", na=False)
].copy()

df_clean[["description"]].head(10)

,description
6,United States : all commercial
7,United States : all industrial
13,New England : all commercial
14,New England : all industrial
20,Connecticut : all commercial
21,Connecticut : all industrial
27,Maine : all commercial
28,Maine : all industrial
34,Massachusetts : all commercial
35,Massachusetts : all industrial


In [5]:
#take out ":" and "all" from sector description
df_clean[["state_or_region", "sector"]] = df_clean["description"].str.split(
    " : ", expand=True
)

df_clean["sector"] = df_clean["sector"].str.replace("all ", "", regex=False)

df_clean[["state_or_region", "sector"]].head(10)

,state_or_region,sector
6,United States,commercial
7,United States,industrial
13,New England,commercial
14,New England,industrial
20,Connecticut,commercial
21,Connecticut,industrial
27,Maine,commercial
28,Maine,industrial
34,Massachusetts,commercial
35,Massachusetts,industrial


In [6]:
#rename source key
df_clean = df_clean.rename(columns={"source key": "source_key"})

df_clean.columns

Index(['description', 'units', 'source_key', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023', '2024', '2025', 'state_or_region', 'sector'],
      dtype='str')

In [7]:
#removing regions to focus on states
regions_to_drop = [
    "United States",
    "New England",
    "Middle Atlantic",
    "East North Central",
    "West North Central",
    "South Atlantic",
    "East South Central",
    "West South Central",
    "Mountain",
    "Pacific Contiguous",
    "Pacific Noncontiguous"
]

df_clean = df_clean[
    ~df_clean["state_or_region"].isin(regions_to_drop)
].copy()

df_clean["state_or_region"].unique()[:20]

<StringArray>
[  'Connecticut',         'Maine', 'Massachusetts', 'New Hampshire',
  'Rhode Island',       'Vermont',    'New Jersey',      'New York',
  'Pennsylvania',      'Illinois',       'Indiana',      'Michigan',
          'Ohio',     'Wisconsin',          'Iowa',        'Kansas',
     'Minnesota',      'Missouri',      'Nebraska',  'North Dakota']
Length: 20, dtype: str

In [ ]:
#Identify year columns
year_cols = [col for col in df_clean.columns if col.isdigit()]

year_cols[:10]

['2001',
 '2002',
 '2003',
 '2004',
 '2005',
 '2006',
 '2007',
 '2008',
 '2009',
 '2010']

In [9]:
df_clean.head()


,description,units,source_key,2001,2002,2003,2004,2005,2006,2007,...,2018,2019,2020,2021,2022,2023,2024,2025,state_or_region,sector
20,Connecticut : all commercial,thousand megawatthours,ELEC.GEN.ALL-CT-96.A,41,48,45,43,40,38,44,...,433,445,330,314,295,294,189,302,Connecticut,commercial
21,Connecticut : all industrial,thousand megawatthours,ELEC.GEN.ALL-CT-97.A,256,311,288,235,207,291,172,...,644,661,614,645,620,655,656,562,Connecticut,industrial
27,Maine : all commercial,thousand megawatthours,ELEC.GEN.ALL-ME-96.A,180,180,183,176,177,172,173,...,165,132,110,98,104,47,41,40,Maine,commercial
28,Maine : all industrial,thousand megawatthours,ELEC.GEN.ALL-ME-97.A,4410,6136,5428,4892,4809,4852,5099,...,1807,1898,1669,1573,1568,1099,1013,1004,Maine,industrial
34,Massachusetts : all commercial,thousand megawatthours,ELEC.GEN.ALL-MA-96.A,558,575,514,573,590,574,503,...,642,606,576,598,1690,1673,1727,1636,Massachusetts,commercial


In [ ]:
#Reshape wide annual value structure to long
total_long = df_clean.melt(
    id_vars=["state_or_region", "sector", "source_key", "units"],
    value_vars=year_cols,
    var_name="year",
    value_name="total_generation"
)

total_long.head(20)

,state_or_region,sector,source_key,units,year,total_generation
0,Connecticut,commercial,ELEC.GEN.ALL-CT-96.A,thousand megawatthours,2001,41
1,Connecticut,industrial,ELEC.GEN.ALL-CT-97.A,thousand megawatthours,2001,256
2,Maine,commercial,ELEC.GEN.ALL-ME-96.A,thousand megawatthours,2001,180
3,Maine,industrial,ELEC.GEN.ALL-ME-97.A,thousand megawatthours,2001,4410
4,Massachusetts,commercial,ELEC.GEN.ALL-MA-96.A,thousand megawatthours,2001,558
5,Massachusetts,industrial,ELEC.GEN.ALL-MA-97.A,thousand megawatthours,2001,409
6,New Hampshire,commercial,ELEC.GEN.ALL-NH-96.A,thousand megawatthours,2001,31
7,New Hampshire,industrial,ELEC.GEN.ALL-NH-97.A,thousand megawatthours,2001,295
8,Rhode Island,commercial,ELEC.GEN.ALL-RI-96.A,thousand megawatthours,2001,50
9,Rhode Island,industrial,ELEC.GEN.ALL-RI-97.A,thousand megawatthours,2001,2


In [13]:
#cleaning generation values
total_long["total_generation"] = total_long["total_generation"].replace("--", pd.NA)

total_long["total_generation"] = pd.to_numeric(
    total_long["total_generation"],
    errors="coerce"
)

total_long.head(20)

,state_or_region,sector,source_key,units,year,total_generation
0,Connecticut,commercial,ELEC.GEN.ALL-CT-96.A,thousand megawatthours,2001,41.0
1,Connecticut,industrial,ELEC.GEN.ALL-CT-97.A,thousand megawatthours,2001,256.0
2,Maine,commercial,ELEC.GEN.ALL-ME-96.A,thousand megawatthours,2001,180.0
3,Maine,industrial,ELEC.GEN.ALL-ME-97.A,thousand megawatthours,2001,4410.0
4,Massachusetts,commercial,ELEC.GEN.ALL-MA-96.A,thousand megawatthours,2001,558.0
5,Massachusetts,industrial,ELEC.GEN.ALL-MA-97.A,thousand megawatthours,2001,409.0
6,New Hampshire,commercial,ELEC.GEN.ALL-NH-96.A,thousand megawatthours,2001,31.0
7,New Hampshire,industrial,ELEC.GEN.ALL-NH-97.A,thousand megawatthours,2001,295.0
8,Rhode Island,commercial,ELEC.GEN.ALL-RI-96.A,thousand megawatthours,2001,50.0
9,Rhode Island,industrial,ELEC.GEN.ALL-RI-97.A,thousand megawatthours,2001,2.0


In [14]:
#now drop rows with NaN values
total_long = total_long.dropna(subset=["total_generation"]).copy()

total_long.shape

(2269, 6)

In [17]:
#reorder and sort columns
total_long = total_long[
    ["state_or_region", "sector", "year", "total_generation", "units", "source_key"]
].sort_values(
    ["state_or_region", "sector", "year"]
).reset_index(drop=True)

total_long.head()

,state_or_region,sector,year,total_generation,units,source_key
0,Alabama,industrial,2001,5858.0,thousand megawatthours,ELEC.GEN.ALL-AL-97.A
1,Alabama,industrial,2002,5365.0,thousand megawatthours,ELEC.GEN.ALL-AL-97.A
2,Alabama,industrial,2003,5266.0,thousand megawatthours,ELEC.GEN.ALL-AL-97.A
3,Alabama,industrial,2004,5227.0,thousand megawatthours,ELEC.GEN.ALL-AL-97.A
4,Alabama,industrial,2005,4650.0,thousand megawatthours,ELEC.GEN.ALL-AL-97.A


In [18]:
print(total_long.head())
print(total_long.shape)
print(total_long.dtypes)
print(total_long["sector"].unique())
print(total_long["state_or_region"].unique()[:20])

  state_or_region      sector  year  total_generation                   units  \
0         Alabama  industrial  2001            5858.0  thousand megawatthours   
1         Alabama  industrial  2002            5365.0  thousand megawatthours   
2         Alabama  industrial  2003            5266.0  thousand megawatthours   
3         Alabama  industrial  2004            5227.0  thousand megawatthours   
4         Alabama  industrial  2005            4650.0  thousand megawatthours   

             source_key  
0  ELEC.GEN.ALL-AL-97.A  
1  ELEC.GEN.ALL-AL-97.A  
2  ELEC.GEN.ALL-AL-97.A  
3  ELEC.GEN.ALL-AL-97.A  
4  ELEC.GEN.ALL-AL-97.A  
(2269, 6)
state_or_region         str
sector                  str
year                    str
total_generation    float64
units                   str
source_key              str
dtype: object
<StringArray>
['industrial', 'commercial']
Length: 2, dtype: str
<StringArray>
[             'Alabama',               'Alaska',              'Arizona',
             

In [19]:
#save
total_long.to_csv("../data/processed/total_generation_clean.csv", index=False)